In [47]:
import pandas as pd
import os
import numpy as np
import openpyxl
from openpyxl.utils.dataframe import dataframe_to_rows
import xlwings as xw

# =====================================================================
# ⚙️ 全局配置区
# =====================================================================
EXCEL_FILE = "CS2_Project.xlsx"
SHEET_RAW = "Raw Data"      # 你的倾倒垃圾桶（中转页）
SHEET_CLEAN = "Clean Data"  # 你的终点站（干净大表）

TEAM_CONFIG = {
    'Vitality':{
    'Mirage': {
        'ropz':  {'T': 'A extremity',   'CT': 'B short'},
        'ZywOo':  {'T': 'AWPer',        'CT': 'AWPer'},
        'apEX':   {'T': 'Roamer',  'CT': 'A connector'},
        'flameZ': {'T': 'Mid Brawl',      'CT': 'A anchor'},
        'mezii':  {'T': 'B extremity',      'CT': 'B anchor'}
    },
    'Dust2': {
        'ropz':  {'T': 'Upper B',   'CT': 'B anchor'},
        'ZywOo':  {'T': 'AWPer',        'CT': 'AWPer'},
        'apEX':   {'T': 'Middle',  'CT': 'CT'},
        'flameZ': {'T': 'Lower B',      'CT': 'Long'},
        'mezii':  {'T': 'Long',      'CT': 'Middle'}
    },
    'Inferno': {
        'ropz':  {'T': 'Apps',   'CT': 'Pit'},
        'ZywOo':  {'T': 'AWPer',        'CT': 'AWPer'},
        'apEX':   {'T': 'Roamer',  'CT': 'B support'},
        'flameZ': {'T': 'Middle',      'CT': 'B Aggressor'},
        'mezii':  {'T': 'Banana',      'CT': 'A Rifle'}
    },
    'Nuke': {
        'ropz':  {'T': 'Lobby',   'CT': 'Ramp'},
        'ZywOo':  {'T': 'AWPer',        'CT': 'AWPer'},
        'apEX':   {'T': 'Outside',  'CT': 'Outside'},
        'flameZ': {'T': 'Silo',      'CT': 'Door'},
        'mezii':  {'T': 'Support',      'CT': 'Floor'}
    },
    'Anubis': {
        'ropz':  {'T': 'A extremity',   'CT': 'A anchor'},
        'ZywOo':  {'T': 'AWPer',        'CT': 'AWPer'},
        'apEX':   {'T': 'Canals',  'CT': 'B connector'},
        'flameZ': {'T': 'Middle',      'CT': 'Middle'},
        'mezii':  {'T': 'B extremity',      'CT': 'B anchor'}
    },
    'Train': {
        'ropz':  {'T': 'Unknown',   'CT': 'A Unknown'},
        'ZywOo':  {'T': 'AWPer',        'CT': 'AWPer'},
        'apEX':   {'T': 'Unknown',  'CT': 'Unknown'},
        'flameZ': {'T': 'Unknown',      'CT': 'Unknown'},
        'mezii':  {'T': 'Unknown',      'CT': 'Unknown'}
    },
    'Overpass': {
        'ropz':  {'T': 'B extremity',   'CT': 'A rifle'},
        'ZywOo':  {'T': 'AWPer',        'CT': 'AWPer'},
        'apEX':   {'T': 'Roamer',  'CT': 'B rotate'},
        'flameZ': {'T': 'A Toilets',      'CT': 'B short'},
        'mezii':  {'T': 'Connector',      'CT': 'B monster'}
    },
    'Ancient': {
        'ropz':  {'T': 'A extremity',   'CT': 'A anchor'},
        'ZywOo':  {'T': 'AWPer',        'CT': 'AWPer'},
        'apEX':   {'T': 'B cave',  'CT': 'B cave'},
        'flameZ': {'T': 'Middle',      'CT': 'Middle'},
        'mezii':  {'T': 'B extremity',      'CT': 'B anchor'}
    }
    },
    'FURIA':{
        'Mirage':{
            'YEKINDAR': {'T': 'Mid Brawl', 'CT': 'B short'},
            'molodoy': {'T': 'AWPer', 'CT': 'AWPer'},
            'FalleN': {'T': 'Roamer', 'CT': 'B anchor'},
            'KSCERATO': {'T': 'B extremity', 'CT': 'A connector'},
            'yuurih': {'T': 'A extremity', 'CT': 'A anchor'}
        },
        'Dust2':{
            'YEKINDAR': {'T': 'Long', 'CT': 'CT'},
            'molodoy': {'T': 'AWPer', 'CT': 'AWPer'},
            'FalleN': {'T': 'Middle', 'CT': 'Long'},
            'KSCERATO': {'T': 'Lower B', 'CT': 'Middle'},
            'yuurih': {'T': 'Upper B', 'CT': 'B anchor'}
        },
        'Inferno':{
            'YEKINDAR': {'T': 'Banana', 'CT': 'B aggressor'},
            'molodoy': {'T': 'AWPer', 'CT': 'AWPer'},
            'FalleN': {'T': 'Roamer', 'CT': 'B support'},
            'KSCERATO': {'T': 'Apps', 'CT': 'A rifle'},
            'yuurih': {'T': 'Middle', 'CT': 'Pit'}
        },
        'Anubis':{
            'YEKINDAR': {'T': 'Canals', 'CT': 'Middle'},
            'molodoy': {'T': 'AWPer', 'CT': 'AWPer'},
            'FalleN': {'T': 'Middle', 'CT': 'B connector'},
            'KSCERATO': {'T': 'A extremity', 'CT': 'A anchor'},
            'yuurih': {'T': 'B extremity', 'CT': 'B anchor'}
        },
        'Nuke':{
           'YEKINDAR': {'T': 'Outside', 'CT': 'Outside'},
            'molodoy': {'T': 'AWPer', 'CT': 'AWPer'},
            'FalleN': {'T': 'Support', 'CT': 'Ramp'},
            'KSCERATO': {'T': 'Lobby', 'CT': 'Door'},
            'yuurih': {'T': 'Silo', 'CT': 'Floor'} 
        },
        'Train':{
            'YEKINDAR': {'T': 'Unknown', 'CT': 'Unknown'},
            'molodoy': {'T': 'AWPer', 'CT': 'AWPer'},
            'FalleN': {'T': 'Unknown', 'CT': 'Unknown'},
            'KSCERATO': {'T': 'Unknown', 'CT': 'Unknown'},
            'yuurih': {'T': 'Unknown', 'CT': 'Unknown'}
        },
        'Ancient':{
            'YEKINDAR': {'T': 'B cave', 'CT': 'B cave'},
            'molodoy': {'T': 'AWPer', 'CT': 'AWPer'},
            'FalleN': {'T': 'A extremity', 'CT': 'A anchor'},
            'KSCERATO': {'T': 'Middle', 'CT': 'Middle'},
            'yuurih': {'T': 'B extremity', 'CT': 'B anchor'}
        },
        'Overpass':{
            'YEKINDAR': {'T': '', 'CT': ''},
            'molodoy': {'T': 'AWPer', 'CT': 'AWPer'},
            'FalleN': {'T': '', 'CT': ''},
            'KSCERATO': {'T': '', 'CT': ''},
            'yuurih': {'T': '', 'CT': ''}
        }
    }
}
MAP_CODE = {
    "1": "Mirage",
    "2": "Dust2",
    "3": "Inferno",
    "4": "Anubis",
    "5": "Nuke",
    "6": "Train",
    "7": "Ancient",
    "8": "Overpass"
}
print("✅ 配置加载成功")

def import_from_queue():
    df_queue = pd.read_excel(
        EXCEL_FILE,
        sheet_name="Import Queue"
    )
    print("沙雕openpyxl擅自把我Data Validation删了！")

    # 找到所有等待处理的比赛
    pending = df_queue[df_queue["Status"] == "Ready"]

    # 如果没有任务
    if pending.empty:
        print("🎉 没有等待导入的比赛！")
        return None

    # 取第一条等待任务
    row = pending.iloc[0]

    # 保存原来的索引（以后更新Status要用）
    current_index = pending.index[0]

    # 读取参数
    team = row["Team"]
    match_name = row["Tournament"]
    stage = row["Stage"]
    opponent = row["Opponent"]
    maps_code = str(row["Maps"])

    # 地图编码转换
    maps = [
        MAP_CODE[c]
        for c in maps_code
    ]

    # 打印检查
    print("========== 当前任务 ==========")
    print(f"Team      : {team}")
    print(f"Match     : {match_name}")
    print(f"Stage     : {stage}")
    print(f"Opponent  : {opponent}")
    print(f"Maps      : {maps}")
    print("==============================")

    print("沙雕openpyxl又擅自把我Data Validation删了！！")

    return (
        df_queue,
        current_index,
        team,
        match_name,
        stage,
        opponent,
        maps
    )

def split_raw_blocks():
    
    df_raw = pd.read_excel(
        EXCEL_FILE,
        sheet_name="Raw Data",
        header=None
    )

    print("沙雕openpyxl又又擅自把我Data Validation删了！！！")

    separator_rows = []

    for i in range(1, len(df_raw)-1):

        if (
            df_raw.iloc[i].isna().all()
            and not df_raw.iloc[i-1].isna().all()
            and not df_raw.iloc[i+1].isna().all()
        ):
            separator_rows.append(i)
    map_ranges = []

    start = 0

    for sep in separator_rows:

        map_ranges.append((start, sep-1))

        start = sep + 1

    map_ranges.append((start, len(df_raw)-1))

    separator_cols = []

    for j in range(1, df_raw.shape[1]-1):

        if (
            df_raw.iloc[:,j].isna().all()
            and not df_raw.iloc[:,j-1].isna().all()
            and not df_raw.iloc[:,j+1].isna().all()
        ):
            separator_cols.append(j)

    blocks = []

    separator_col = separator_cols[0]

    for start,end in map_ranges:

        block = df_raw.iloc[start:end+1]

        t_block = block.iloc[:, :separator_col]
        t_block.columns = range(t_block.shape[1])

        ct_block = block.iloc[:, separator_col+1:]
        ct_block.columns = range(ct_block.shape[1])

        blocks.append({
            "T": t_block,
            "CT": ct_block
        })

    return blocks

# =====================================================
# 列配置（以后HLTV改列顺序只改这里）
# =====================================================

SIDE_COLUMNS = {
    "player": 0,
    "swing": 2,
    "adr": 3,
    "rating": 5
}

# =====================================================
# 清洗一侧(T或CT)的数据
# 输入：blocks[i]["T"] 或 blocks[i]["CT"]
# 输出：Player Rating ADR Swing
# =====================================================

def extract_player_id(name):
    name = str(name)
    if name.count("'") >= 2:
        start = name.find("'") + 1
        end = name.find("'", start)
        return name[start:end]
    return name.strip()

def clean_side_block(side_block):
    print(side_block.columns)

    df_clean = pd.DataFrame(
        columns=[
            "Player",
            "Rating",
            "ADR",
            "Swing"
        ]
    )

    # 每两行是一位选手
    for i in range(1, len(side_block)-1, 2):
        player_row = side_block.iloc[i+1]

        stat_row = side_block.iloc[i]

        player = extract_player_id(
            player_row[SIDE_COLUMNS["player"]]
        )

        rating = stat_row[SIDE_COLUMNS["rating"]]

        adr = stat_row[SIDE_COLUMNS["adr"]]

        swing = stat_row[SIDE_COLUMNS["swing"]]

        df_clean.loc[len(df_clean)] = [
            player,
            rating,
            adr,
            swing
        ]

    if len(df_clean) != 5:
        print("⚠️ 本地图识别到的选手数量异常。")

    return df_clean

def clean_cs2_match():
    if not os.path.exists(EXCEL_FILE):
        print(f"❌ 错误：未找到文件 【{EXCEL_FILE}】")
        return
    result = import_from_queue()

    if result is None:
        return
    (
        df_queue,
        current_index,
        team,
        match_name,
        stage,
        opponent,
        maps
    ) = result
    blocks = split_raw_blocks()

    if len(blocks) != len(maps):
        print("❌ Raw Data里的地图数量和Import Queue不一致!")
        print(f"Queue: {len(maps)}")
        print(f"Raw Data: {len(blocks)}")
        return
    
    all_new_rows = []
    
    # 读取历史大表（Clean Data）
    try:
        df_old_master = pd.read_excel(EXCEL_FILE, sheet_name=SHEET_CLEAN)
    except Exception:
        print("❌ 错误：无法读取到历史大表。")
        return

    for map_name, block in zip(maps, blocks):
        print(f"========== {map_name} ==========")
        t_df = clean_side_block(
            block["T"]
        )
        ct_df = clean_side_block(
            block["CT"]
        )
        merged = pd.merge(
            t_df,
            ct_df,
            on="Player",
            suffixes=(
                "_T",
                "_CT"
            )
        )
        t_positions = [
            TEAM_CONFIG[team][map_name][player]["T"]
            for player in merged["Player"]
        ]
        ct_positions = [
            TEAM_CONFIG[team][map_name][player]["CT"]
            for player in merged["Player"]
        ]
        df_new = pd.DataFrame({
            "Match": match_name,
            "Stage": stage,
            "Team": team,
            "Opponent": opponent,
            "Player": merged["Player"],
            "Map": map_name,
            "T-side Position": t_positions,
            "CT-side Position": ct_positions,
            "T-side Rating": merged["Rating_T"],
            "CT-side Rating": merged["Rating_CT"],
            "T-side ADR": merged["ADR_T"],
            "CT-side ADR": merged["ADR_CT"],
            "T-side Round Swing": merged["Swing_T"],
            "CT-side Round Swing": merged["Swing_CT"]
        })
        print(f"✅ {map_name} 完成")
        all_new_rows.append(df_new)
    df_all_new = pd.concat(
        all_new_rows,
        ignore_index=True
    )
        # print(merged)
    # print(df_all_new)

    df_updated_master = pd.concat(
        [df_old_master, df_all_new],
        ignore_index=True
    )
    df_queue.loc[current_index, "Status"] = "Done"

    app = xw.App(visible=False)
    wb = xw.Book(EXCEL_FILE)
    try:
        ws_clean = wb.sheets[SHEET_CLEAN]
        ws_clean.clear_contents()
        ws_clean.range("A1").options(pd.DataFrame, index=False).value = df_updated_master

        ws_queue = wb.sheets["Import Queue"]
        ws_queue.range(f"A{current_index + 2}").value = "Done"

        ws_raw = wb.sheets[SHEET_RAW]
        ws_raw.clear_contents()
        wb.save()
    finally:
        wb.close()
        app.quit()

    print(f"🎉 成功导入 {len(maps)} 张地图，共 {len(df_all_new)} 条记录！")
    return df_all_new
clean_cs2_match()


# Maps: Dust2, Mirage, Inferno, Ancient, Nuke, Anubis, Train, Overpass
# Teams: Vitality, Falcons, Liquid, MOUZ, Spirit, FaZe, Natus Vincere, The MongolZ, G2, GamerLegion, Virtus.pro, Astralis, FURIA, Legacy, Aurora, PARIVISION, FUT, 
# Stages: Group stage double elimination, Single-elimination, Upper bracket final, Lower bracket decider, Playoff, Grand final, Swiss Round, Major stage 1 Swiss round, Major stage 2 Swiss round, Major stage 3 Swiss round, Major playoff, Major grand final, 3rd place decider
# Stage System 1: Swiss rounds (Swiss Round)
# Stage System 2: IEM (Group stage double elimination, Upper bracket final / Qualification match, Lower bracket decider)
# Stage System 3: BLAST (Group stage, Qualification Match， Elimination Match)
# Stage System 4: All single elimination (Single-elimination)
# Stage System 5: Major (Major stage 1 Swiss round, Major stage 2 Swiss round, Major stage 3 Swiss round, Major playoff, Major grand final)
# Shared stages: Playoff, Grand final, 3rd place decider

✅ 配置加载成功
沙雕openpyxl擅自把我Data Validation删了！
========== 当前任务 ==========
Team      : FURIA
Match     : IEM Cologne Major 2026
Stage     : Major grand final
Opponent  : Falcons
Maps      : ['Mirage', 'Anubis', 'Inferno']
沙雕openpyxl又擅自把我Data Validation删了！！
沙雕openpyxl又又擅自把我Data Validation删了！！！


c:\ProgramData\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:79: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


========== Mirage ==========
RangeIndex(start=0, stop=6, step=1)
RangeIndex(start=0, stop=6, step=1)
✅ Mirage 完成
========== Anubis ==========
RangeIndex(start=0, stop=6, step=1)
RangeIndex(start=0, stop=6, step=1)
✅ Anubis 完成
========== Inferno ==========
RangeIndex(start=0, stop=6, step=1)
RangeIndex(start=0, stop=6, step=1)
✅ Inferno 完成
🎉 成功导入 3 张地图，共 15 条记录！


,Match,Stage,Team,Opponent,Player,Map,T-side Position,CT-side Position,T-side Rating,CT-side Rating,T-side ADR,CT-side ADR,T-side Round Swing,CT-side Round Swing
0,IEM Cologne Major 2026,Major grand final,FURIA,Falcons,molodoy,Mirage,AWPer,AWPer,1.51,1.32,75.1,72.3,0.0601,0.0198
1,IEM Cologne Major 2026,Major grand final,FURIA,Falcons,YEKINDAR,Mirage,Mid Brawl,B short,1.13,0.76,78.4,52.2,0.0145,-0.0456
2,IEM Cologne Major 2026,Major grand final,FURIA,Falcons,KSCERATO,Mirage,B extremity,A connector,0.93,1.03,67.4,84.9,-0.0637,-0.0158
3,IEM Cologne Major 2026,Major grand final,FURIA,Falcons,yuurih,Mirage,A extremity,A anchor,0.78,0.72,38.9,62.1,-0.0236,-0.0153
4,IEM Cologne Major 2026,Major grand final,FURIA,Falcons,FalleN,Mirage,Roamer,B anchor,0.66,1.05,43.3,87.4,-0.0400,-0.0103
5,IEM Cologne Major 2026,Major grand final,FURIA,Falcons,YEKINDAR,Anubis,Canals,Middle,1.35,0.61,103.2,63.3,0.0152,-0.0829
6,IEM Cologne Major 2026,Major grand final,FURIA,Falcons,molodoy,Anubis,AWPer,AWPer,1.23,0.70,55.8,57.6,0.0686,-0.0357
7,IEM Cologne Major 2026,Major grand final,FURIA,Falcons,KSCERATO,Anubis,A extremity,A anchor,1.14,0.36,59.7,32.3,0.0316,-0.1281
8,IEM Cologne Major 2026,Major grand final,FURIA,Falcons,FalleN,Anubis,Middle,B connector,1.07,1.29,75.8,94.2,-0.0221,0.0384
9,IEM Cologne Major 2026,Major grand final,FURIA,Falcons,yuurih,Anubis,B extremity,B anchor,0.75,0.74,71.0,65.0,-0.0461,-0.0874
